# JaxFormer on Kaggle TPU v5e-8

Thin driver over the public repo. It (1) confirms 8 TPU chips, (2) produces the TPU
row of the cross-hardware benchmark with the *same* `run_jax()` used on CPU, and
(3) optionally runs a short real training loop if a tokenized-corpus dataset is
mounted.

**Setup:** Notebook settings -> Accelerator -> **TPU VM v3-8 / v5e-8**. Internet **On**
(needed for the pip install). No training logic lives here — it all comes from the
installed package, so the TPU numbers are comparable to the GPU/CPU ones by
construction.

In [ ]:
# JAX with TPU support + the repo (framework-free config, JAX model, bench).
!pip -q install "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
!pip -q install git+https://github.com/fishy-ops/jaxformer.git
# The bench/ and torch_ref/ trees are not packaged; grab the source for `bench.bench_training`.
!git clone -q https://github.com/fishy-ops/jaxformer.git /kaggle/working/jaxformer || true

In [ ]:
import sys, jax
sys.path.insert(0, "/kaggle/working/jaxformer")  # for the bench/ package
devices = jax.devices()
print(len(devices), "devices:", devices[0].platform, devices[0].device_kind)
assert devices[0].platform == "tpu", "Accelerator is not a TPU — set it in Notebook settings"
print("OK: ", len(devices), "TPU chips")

In [ ]:
# TPU row of the cross-hardware table. run_jax() auto-detects the TPU host, computes in
# bf16, shards the batch across all 8 chips, and reports the same three metrics.
import json
from bench.bench_training import run_jax

row = run_jax(batch=8, seq=256, train_steps=20, infer_tokens=32)
print(json.dumps(row, indent=2))

# Save it, then download and drop into the repo as
# bench/results/training_jax_tpu_kaggle.json so `python -m bench.plots` picks it up.
with open("/kaggle/working/training_jax_tpu_kaggle.json", "w") as f:
    json.dump(row, f, indent=2)

## Optional: a short real training run

For an actual loss curve rather than a throughput number, prepare the corpus locally
(`python scripts/prepare_data.py --out-dir data`), upload `data/` as a private Kaggle
Dataset, mount it, and point `TokenDataset` at it. Budget the run to finish inside one
~9 h session with checkpoint-resume so a timeout costs at most one checkpoint interval.
Sanity target: val loss ≈ 3.2–3.6 nats; a plateau much above 4.0 means the bug is
upstream (tokenizer or loader), not the optimizer.